# Fine-tune FunctionGemma 270M for Ari

Based on [Google's FunctionGemma fine-tuning notebook](https://github.com/google-gemma/cookbook/blob/main/FunctionGemma/%5BFunctionGemma%5DFinetune_FunctionGemma_270M_for_Mobile_Actions_with_Hugging_Face.ipynb).

Changes from Google's version:
- Uses our combined dataset (Ari skills + mobile actions + negatives)
- Exports to GGUF Q4_K_M for llama.cpp

**Runtime:** Colab T4 GPU (~20 minutes training)

In [ ]:
!pip install -q torch transformers trl datasets huggingface_hub gguf

In [ ]:
from huggingface_hub import login
login()  # Paste your HF token (needs Gemma access)

## Upload dataset

Upload `ari-functiongemma-dataset.jsonl` from your local machine.
Generate it with: `python3 tools/generate-functiongemma-dataset.py > ari-functiongemma-dataset.jsonl`

In [ ]:
from google.colab import files
uploaded = files.upload()  # Upload ari-functiongemma-dataset.jsonl
DATASET_FILE = list(uploaded.keys())[0]
print(f'Uploaded: {DATASET_FILE}')

## Load model and dataset

In [ ]:
import json
import torch
from transformers import AutoProcessor, AutoModelForCausalLM

GEMMA_MODEL_ID = "google/functiongemma-270m-it"

processor = AutoProcessor.from_pretrained(GEMMA_MODEL_ID, device_map="auto")
base_model = AutoModelForCausalLM.from_pretrained(
    GEMMA_MODEL_ID, dtype=torch.bfloat16, device_map="auto",
    attn_implementation='eager',
)
base_model.config.pad_token_id = processor.pad_token_id
print(f'Model loaded: {base_model.num_parameters():,} parameters')

In [ ]:
# Load and split dataset
samples = []
with open(DATASET_FILE) as f:
    for line in f:
        samples.append(json.loads(line))

train_data = [s for s in samples if s['metadata'] == 'train']
eval_data = [s for s in samples if s['metadata'] == 'eval']
print(f'Train: {len(train_data)}, Eval: {len(eval_data)}')

In [ ]:
# Convert to prompt/completion format using the chat template
def apply_format(sample):
    messages = sample['messages']
    tools = sample['tools']

    prompt_and_completion = processor.apply_chat_template(
        messages, tools=tools, tokenize=False,
        add_generation_prompt=False,
    )

    prompt = processor.apply_chat_template(
        messages[:-1], tools=tools, tokenize=False,
        add_generation_prompt=True,
    )

    completion = prompt_and_completion[len(prompt):]
    return {'prompt': prompt, 'completion': completion}

train_formatted = [apply_format(s) for s in train_data]
eval_formatted = [apply_format(s) for s in eval_data]

# Find max token length for training config
max_len = max(
    len(processor.encode(s['prompt'] + s['completion']))
    for s in train_formatted[:500]
)
MAX_SEQ_LEN = min(max_len + 100, 2048)
print(f'Max sequence length: {MAX_SEQ_LEN}')
print(f'Sample completion: {train_formatted[0]["completion"][:200]}')

In [ ]:
from datasets import Dataset

train_ds = Dataset.from_list(train_formatted)
eval_ds = Dataset.from_list(eval_formatted)
print(train_ds)
print(eval_ds)

## Train

In [ ]:
from trl import SFTTrainer, SFTConfig

training_config = SFTConfig(
    output_dir="./ari-functiongemma-finetuned",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=1e-5,
    lr_scheduler_type="cosine",
    max_seq_length=MAX_SEQ_LEN,
    gradient_checkpointing=True,
    packing=False,
    optim="adamw_torch_fused",
    bf16=True,
    completion_only_loss=True,
    logging_strategy="steps",
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="epoch",
    report_to="none",
)

trainer = SFTTrainer(
    model=base_model,
    args=training_config,
    processing_class=processor,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
)

trainer.train()

In [ ]:
# Save the fine-tuned model
trainer.save_model("./ari-functiongemma-finetuned")
processor.save_pretrained("./ari-functiongemma-finetuned")
print('Model saved.')

## Quick eval

In [ ]:
import re

test_cases = [
    ("what time is it", "current_time"),
    ("what's the date today", "current_date"),
    ("calculate 5 + 3", "calculator"),
    ("hello", "greeting"),
    ("open spotify", "open_app"),
    ("search for python tutorials", "search"),
    ("flip a coin", "coin_flip"),
    ("do you know what hour it is", "current_time"),
    ("launch my music player", "open_app"),
    ("how much is fifteen percent of two hundred", "calculator"),
    ("find me information about black holes", "search"),
    ("let's leave it to chance, heads or tails", "coin_flip"),
    ("what is the capital of France", None),
    ("tell me a joke", None),
    ("why is the sky blue", None),
]

# Build tools for eval
ari_skills = [
    {"type": "function", "function": {"name": "current_time", "description": "Tells the current time. Use when the user asks what time it is, what hour it is, whether it is morning or afternoon, or anything about the current time of day.", "parameters": {"type": "object", "properties": {}}}},
    {"type": "function", "function": {"name": "current_date", "description": "Tells today's date. Use when the user asks what day it is, what date it is, which day of the week it is, or anything about today's date.", "parameters": {"type": "object", "properties": {}}}},
    {"type": "function", "function": {"name": "calculator", "description": "Evaluates math expressions. Use when the user asks to calculate, compute, or figure out any mathematical expression, percentage, division, multiplication, addition, subtraction, or arithmetic.", "parameters": {"type": "object", "properties": {"expression": {"type": "string", "description": "The math expression to evaluate."}}}}},
    {"type": "function", "function": {"name": "greeting", "description": "Responds to greetings. Use when the user says hello, hi, hey, good morning, good evening, howdy, what's up, or asks how Ari is doing.", "parameters": {"type": "object", "properties": {}}}},
    {"type": "function", "function": {"name": "open_app", "description": "Opens or launches apps by name. Use when the user asks to open, launch, start, run, or fire up an application or app.", "parameters": {"type": "object", "properties": {"app_name": {"type": "string", "description": "Name of the app to open."}}}}},
    {"type": "function", "function": {"name": "search", "description": "Searches the web. Use when the user asks to search, look up, find information about something, or google something.", "parameters": {"type": "object", "properties": {"query": {"type": "string", "description": "The search query."}}}}},
    {"type": "function", "function": {"name": "coin_flip", "description": "Flips a virtual coin and returns heads or tails. Use when the user asks to flip a coin, toss a coin, or make a random heads or tails choice.", "parameters": {"type": "object", "properties": {}}}},
]

skill_names = {t['function']['name'] for t in ari_skills}
correct = 0

for user_input, expected in test_cases:
    messages = [
        {"role": "developer", "content": "You are a model that can do function calling with the following functions"},
        {"role": "user", "content": user_input},
    ]
    inputs = processor.apply_chat_template(
        messages, tools=ari_skills, add_generation_prompt=True,
        return_dict=True, return_tensors="pt",
    )
    out = base_model.generate(
        **inputs.to(base_model.device),
        pad_token_id=processor.eos_token_id,
        max_new_tokens=60,
        temperature=1e-3,
        do_sample=True,
    )
    generated = processor.decode(out[0][len(inputs['input_ids'][0]):], skip_special_tokens=False)
    
    m = re.search(r'<start_function_call>call:(\w+)\{', generated)
    got = m.group(1) if m and m.group(1) in skill_names else None
    
    ok = (expected is None and got is None) or (expected is not None and got == expected)
    correct += ok
    mark = 'PASS' if ok else 'FAIL'
    print(f'[{mark}] {user_input:50s} expected={str(expected):15s} got={str(got):15s} raw={generated[:80]}')

print(f'\nScore: {correct}/{len(test_cases)} ({100*correct/len(test_cases):.0f}%)')

## Convert to GGUF

In [ ]:
!git clone --depth 1 https://github.com/ggml-org/llama.cpp.git
!pip install -q gguf

In [ ]:
!python3 llama.cpp/convert_hf_to_gguf.py ./ari-functiongemma-finetuned \
    --outfile ari-functiongemma-f16.gguf --outtype f16

In [ ]:
# Build llama-quantize
!cd llama.cpp && cmake -B build && cmake --build build --target llama-quantize -j4

In [ ]:
!llama.cpp/build/bin/llama-quantize ari-functiongemma-f16.gguf ari-functiongemma-q4_k_m.gguf Q4_K_M

In [ ]:
import os
size_mb = os.path.getsize('ari-functiongemma-q4_k_m.gguf') / 1024 / 1024
print(f'Final model: {size_mb:.0f} MB')

files.download('ari-functiongemma-q4_k_m.gguf')